In [1]:
from actor_system import ActorSystem, PipelinedActor
from msg_type import PipelineDone, BufferRelease, MIBDecoded, Token, MIBFail, Config

In [2]:
class MIBDecodeActor(PipelinedActor):

    ANT_TRIALS = [1, 2, 4]

    def __init__(self, system, buf, pool, graph, node_pipeline):
        # node_pipeline = [crs_node, pbch_node, bch_node]
        super().__init__('mib_decode', system, pool, graph, node_pipeline[0])
        self._buf = buf
        self._pipeline = node_pipeline
        self._known_n_ant = None             # cached after first success
        self._trial_slots = {}               # slot_idx → trial_idx (active blind trials)

    # ---- Override: keep slot alive during blind trial ----

    def _default_behavior(self, msg):
        if isinstance(msg, PipelineDone):
            # on_result returns True = retry (keep slot), False = done (release slot)
            keep = self.on_result(msg)
            if not keep:
                self._pool.release(msg.slot)
                self._in_flight -= 1
            self._try_dispatch()
        elif isinstance(msg, Config):
            self.on_config(msg)
        else:
            # DataReady handled by PipelinedActor (queue + dispatch)
            super()._default_behavior(msg)

    # ---- Fill slot from buffer ----

    def _fill_slot(self, slot_idx, msg):
        slot = self._pool.slots[slot_idx]
        data = self._buf.read_protected(msg.pid)
        slot.data[:len(data)] = data
        slot.tag = msg.tag

        # tag carries cell info from cell search: N_id, f_d
        slot.N_id = msg.tag['N_id']
        slot.f_d = msg.tag['f_d']
        self.system.send_message('buffer_manager', BufferRelease(pid=msg.pid))

        if self._known_n_ant:
            # fast path: use cached antenna count
            slot.n_ant = self._known_n_ant
        else:
            # blind trial: start with first hypothesis
            slot.n_ant = self.ANT_TRIALS[0]
            self._trial_slots[slot_idx] = 0

    # ---- Result handling with blind trial ----

    def on_result(self, msg):
        slot = self._pool.slots[msg.slot]

        if slot.mib_decoded:
            # ---- SUCCESS: cache n_ant, report to controller ----
            self._known_n_ant = slot.n_ant
            self._trial_slots.pop(msg.slot, None)
            self.system.send_message('controller', MIBDecoded(
                dl_bw=slot.dl_bw, n_ant=slot.n_ant,
                sfn=slot.sfn, phich_dur=slot.phich_dur,
                phich_res=slot.phich_res, tag=msg.tag))
            return False  # release slot

        elif msg.slot in self._trial_slots:
            # ---- BLIND TRIAL: try next n_ant hypothesis ----
            trial_idx = self._trial_slots[msg.slot] + 1

            if trial_idx < len(self.ANT_TRIALS):
                # more hypotheses to try — re-enter pipeline at CRS
                self._trial_slots[msg.slot] = trial_idx
                slot.n_ant = self.ANT_TRIALS[trial_idx]
                slot.mib_decoded = False
                self._graph.try_put(self._first_node, Token(slot=msg.slot, tag=msg.tag))
                return True  # keep slot — still in use
            else:
                # all hypotheses exhausted
                self._trial_slots.pop(msg.slot, None)
                self.system.send_message('controller', MIBFail(tag=msg.tag))
                return False  # release slot

        else:
            # ---- CACHED n_ant FAILED: restart blind trial ----
            # CRS is still valid — re-enter at CRS with first hypothesis
            self._known_n_ant = None
            slot.n_ant = self.ANT_TRIALS[0]
            slot.mib_decoded = False
            self._trial_slots[msg.slot] = 0
            self._graph.try_put(self._first_node, Token(slot=msg.slot, tag=msg.tag))
            return True  # keep slot — restarting trial

    # ---- Config: propagate to pipeline stages ----

    def on_config(self, msg):
        for node in self._pipeline:
            if node.name in msg.params:
                node.func.config(**msg.params[node.name])

#### Test
1. cell search actor 

In [3]:
from data.lte_system_info import LTEParams
from flow_graph import FlowGraph, FunctionNode, ResizableSlotPool, SlotPool
from cell_search import PSSDetection, SSSDetection
import time
from actor_system import Actor
from msg_type import PipelineDone
import numpy as np
from slot_type import CellSearchSlot
from buffer_manager_actor import BufferManagerActor
from cell_search_actor import CellSearchActor
from msg_type import BufferRead, CellFound, NoCell

In [4]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [5]:
class CollectorActor(Actor):
    def __init__(self, name, system):
        super().__init__(name, system)
        self.messages = []
    def _default_behavior(self, message):
        self.messages.append(message)

system = ActorSystem()
graph = FlowGraph(num_workers=4)

pool = ResizableSlotPool(6, lambda: CellSearchSlot(params.N_subframe))

pss_func = PSSDetection(params, peak_ratio=5.0)
sss_func = SSSDetection(params, peak_ratio=8.0)

pss_node = FunctionNode('pss', pool.make_stage(pss_func), concurrency=1)
sss_node = FunctionNode('sss', pool.make_stage(sss_func), concurrency=1,
                        done_callback=lambda token: system.send_message(
                            'cell_search', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(pss_node, sss_node)

ctrl = CollectorActor('controller', system)
bm = BufferManagerActor(system, rxf, buffer_size=len(rxf),
                        batch_size=len(rxf), ingest_delay=0.0)
cs = CellSearchActor(system, bm.buf, pool, graph, params, [pss_node, sss_node])

system.create_actor(ctrl)
system.create_actor(cs)
system.create_actor(bm)

system.send_message('buffer_manager', 'start')

# ---- initial search ----
stride = params.stride
pos = 0
chunk_id = 0
while pos + params.N_subframe <= params.N_half_frame:
    system.send_message('buffer_manager', BufferRead(
        offset=pos, length=params.N_subframe, dest='cell_search',
        tag={'mode': 'initial_search', 'chunk_tag': chunk_id, 'pos': pos}))
    pos += stride
    chunk_id += 1

time.sleep(0.5)

# ---- tracking: one half-frame later ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
msg = cell_found[0]
pss_global = msg.tag['pos'] + msg.pss_local_index

expected_pss = pss_global + params.N_half_frame
track_start = expected_pss - 2 * params.N_ofdm_sym
track_len = params.pss_tracking_len
track_offset = track_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=track_offset, length=track_len, dest='cell_search',
    tag={'mode': 'tracking', 'frame_tag': 0}))

time.sleep(0.5)

# ---- print results ----
print(f'\nController received {len(ctrl.messages)} messages:')
for m in ctrl.messages:
    if isinstance(m, CellFound):
        if m.tag.get('mode') == 'initial_search':
            pss_g = m.tag['pos'] + m.pss_local_index
            print(f'  CellFound (initial): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
        elif m.tag.get('mode') == 'tracking':
            pss_g = track_start + m.pss_local_index
            print(f'  CellFound (tracking): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
    elif isinstance(m, NoCell):
        print(f'  NoCell: {m.tag.get("mode")}')
    else:
        print(f'  {m}')


Controller received 5 messages:
  start
  NoCell: initial_search
  NoCell: initial_search
  CellFound (initial): PCI=380, F=0, f_d=1126.9, pss_global=36043
  CellFound (tracking): PCI=380, F=1, f_d=1128.4, pss_global=112842


2. mib decoding actor

In [6]:
from mib_decode import PBCHDecoding, BCHDecoding
from crs_estimate import CRSChannelEstimation
from slot_type import MIBSlot

In [7]:
# ---- extract cell search results ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
msg = cell_found[0]
N_id = msg.N_id
f_d = msg.f_d
pss_global = msg.tag['pos'] + msg.pss_local_index

print(f"Cell found: PCI={N_id}, f_d={f_d:.1f} Hz, pss_global={pss_global}")

# ---- MIB pipeline on SAME graph ----
mib_pool = SlotPool(4, lambda: MIBSlot(params.pbch_len))

crs_func = CRSChannelEstimation(params)
pbch_func = PBCHDecoding()
bch_func = BCHDecoding()

crs_node  = FunctionNode('mib_crs',  mib_pool.make_stage(crs_func),  concurrency=1)
pbch_node = FunctionNode('mib_pbch', mib_pool.make_stage(pbch_func), concurrency=1)
bch_node  = FunctionNode('mib_bch',  mib_pool.make_stage(bch_func),  concurrency=1,
                         done_callback=lambda token: system.send_message(
                             'mib_decode', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(crs_node, pbch_node)
graph.add_edge(pbch_node, bch_node)

mib = MIBDecodeActor(system, bm.buf, mib_pool, graph, [crs_node, pbch_node, bch_node])
system.create_actor(mib)

# ---- config ----
system.send_message('mib_decode', Config(params={
    'mib_crs':  {'N_id': N_id, 'N_rb': 6, 'ns': 1,
                 'n_symbols': 4, 'is_slot_start': True},
    'mib_pbch': {'N_id': N_id},
    'mib_bch':  {'N_id': N_id}
}))
time.sleep(0.1)

# ==== Step 1: first MIB decode (blind trial) ====

slot1_start = pss_global + params.N_FFT
slot1_offset = slot1_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=slot1_offset, length=params.pbch_len, dest='mib_decode',
    tag={'N_id': N_id, 'f_d': f_d, 'frame_tag': 0}))

time.sleep(1)

mib_msgs = [m for m in ctrl.messages if isinstance(m, MIBDecoded)]
mib_fails = [m for m in ctrl.messages if isinstance(m, MIBFail)]
print(f'\nMIBDecoded: {len(mib_msgs)}, MIBFail: {len(mib_fails)}')
assert len(mib_msgs) == 1, f"Expected 1 MIBDecoded, got {len(mib_msgs)}"

r = mib_msgs[0]
print(f"\n=== MIB #1 (blind trial) ===")
print(f"  n_ant={r.n_ant}, BW={r.dl_bw}, SFN={r.sfn}")
print(f"  PHICH dur={r.phich_dur}, res={r.phich_res}")
print(f"  cached n_ant = {mib._known_n_ant}")

# ==== Step 2: tracking PSS for next even-SFN frame ====

sfn = r.sfn
delta_frames = 2 if sfn % 2 == 0 else 1
next_even_pss = pss_global + delta_frames * params.N_frame

track_start = next_even_pss - 2 * params.N_ofdm_sym
track_len = params.pss_tracking_len
track_offset = track_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=track_offset, length=track_len, dest='cell_search',
    tag={'mode': 'tracking', 'frame_tag': 1}))

time.sleep(0.5)

tracking_found = [m for m in ctrl.messages if isinstance(m, CellFound)
                  and m.tag.get('mode') == 'tracking'
                  and m.tag.get('frame_tag') == 1]
assert len(tracking_found) == 1, f"Expected 1 tracking CellFound, got {len(tracking_found)}"

track_pss_global = track_start + tracking_found[0].pss_local_index
track_f_d = tracking_found[0].f_d

print(f"\n=== Tracking PSS ===")
print(f"  pss_pos={track_pss_global}, "
      f"drift={track_pss_global - next_even_pss}, f_d={track_f_d:.1f} Hz")

# ==== Step 3: second MIB decode (fast path) ====

slot1_start_2 = track_pss_global + params.N_FFT
mib_offset_2 = slot1_start_2 - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=mib_offset_2, length=params.pbch_len, dest='mib_decode',
    tag={'N_id': N_id, 'f_d': track_f_d, 'frame_tag': 1}))

time.sleep(0.5)

mib_msgs_2 = [m for m in ctrl.messages if isinstance(m, MIBDecoded)]
assert len(mib_msgs_2) == 2, f"Expected 2 MIBDecoded, got {len(mib_msgs_2)}"

r2 = mib_msgs_2[1]
print(f"\n=== MIB #2 (fast path) ===")
print(f"  n_ant={r2.n_ant}, BW={r2.dl_bw}, SFN={r2.sfn}")
print(f"  PHICH dur={r2.phich_dur}, res={r2.phich_res}")

graph.shutdown()  # shutdown only here

Cell found: PCI=380, f_d=1126.9 Hz, pss_global=36043

MIBDecoded: 1, MIBFail: 0

=== MIB #1 (blind trial) ===
  n_ant=2, BW=50, SFN=313
  PHICH dur=normal, res=1
  cached n_ant = 2

=== Tracking PSS ===
  pss_pos=189643, drift=0, f_d=1164.2 Hz

=== MIB #2 (fast path) ===
  n_ant=2, BW=50, SFN=314
  PHICH dur=normal, res=1
